In [82]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
from astropy.visualization import simple_norm
from astropy.modeling.models import Sersic2D
from photutils.segmentation import detect_threshold, detect_sources
import time
from astropy.visualization import simple_norm
import statmorph
from statmorph.utils.image_diagnostics import make_figure
%matplotlib inline

# Python interpreter path: /nvme/scratch/software/anaconda3/envs/statmorph_env/bin/python


with fits.open("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits") as hdul:
    data = hdul[1].data
TABLE = Table(data)
GALAXY_ID = TABLE["SURVEY_ID"]   # object ID, used for labeling/output files
SURVEY = TABLE["SURVEY"]

FILTERS = ["F444W", "F356W", "F277W"]  # filters to fit, used for labeling/output files

In [83]:
def read_fits_path(path, ext):

    path = Path(path)

    if not path.exists():
        print(f"{path} not found.")
        return None

    try:
        with fits.open(path) as hdul:
            return hdul[ext].data
    except Exception as e:
        print(f"Could not open {path}: {e}")
        return None

In [84]:
def load_science_data(galaxy_id, filt):
    """
    Load science image, segmentation-derived mask, rms map, and psf.
    """
    science_fits_path = f"/nvme/scratch/work/alberttg/Summer_project/Cutouts_3.0as/{galaxy_id}/cutouts/{galaxy_id}_science_{filt}.fits"
    seg_fits_path = f"/nvme/scratch/work/alberttg/Summer_project/Cutouts_3.0as/{galaxy_id}/cutouts/{galaxy_id}_segmentation_{filt}.fits"
    rms_fits_path = f"/nvme/scratch/work/alberttg/Summer_project/Cutouts_3.0as/{galaxy_id}/cutouts/{galaxy_id}_error_{filt}.fits"

    image = read_fits_path(science_fits_path, ext=0)
    segmap = read_fits_path(seg_fits_path, ext=0)
    rms = read_fits_path(rms_fits_path, ext=0)

    if image.shape != segmap.shape or image.shape != rms.shape:
        raise ValueError(
            f"Shape mismatch: image {image.shape}, segmap {segmap.shape}, "
            f"rms {rms.shape}. All three extensions must match."
        )

    # Build a mask from the segmentation map.
    # Convention: segmap == 0 is background (good/unmasked), any nonzero
    # segmentation ID marks a source. We assume the central/target source
    # sits at the ID found at the image center, and mask out all *other*
    # nonzero segmentation IDs (neighboring sources), while leaving the
    # target source itself unmasked.
    mask = build_mask_from_segmap(segmap)

    return image, mask, rms, segmap


def build_mask_from_segmap(segmap):
    """
    Convert a segmentation map into a boolean mask suitable for pysersic,
    where True = pixel should be masked/ignored (bad pixel or nearby
    contaminating source), and False = good pixel to use in the fit.

    Assumes the target galaxy is the segmentation ID present at the center
    of the cutout. All other nonzero IDs are masked out; background (0)
    stays unmasked.
    """
    ny, nx = segmap.shape
    cy, cx = ny // 2, nx // 2
    center_id = segmap[cy, cx]

    if center_id == 0:
        # Center pixel is background; search a small box around the center
        # for the nearest nonzero segmentation ID to use as the target.
        box = 5
        y0, y1 = max(0, cy - box), min(ny, cy + box + 1)
        x0, x1 = max(0, cx - box), min(nx, cx + box + 1)
        sub = segmap[y0:y1, x0:x1]
        nonzero = sub[sub != 0]
        if nonzero.size > 0:
            vals, counts = np.unique(nonzero, return_counts=True)
            center_id = vals[np.argmax(counts)]
        else:
            center_id = 0  # give up, nothing to mask as "target"

    # Mask everything that is a source (nonzero) and is NOT the target ID.
    mask = (segmap != 0) & (segmap != center_id)
    return mask


def load_and_crop_psf(psf_fits_path, science_image_shape, psf_ext=0):
    """
    Load the PSF from a FITS file and crop it (centered) so its dimensions
    are odd and no larger than the science image, which is the standard
    requirement for pysersic's convolution.
    """
    with fits.open(psf_fits_path) as hdul:
        psf = hdul[psf_ext].data.astype(float)

    psf = crop_to_odd(psf)

    # Also ensure PSF isn't larger than the science image itself.
    max_ny, max_nx = science_image_shape
    py, px = psf.shape
    target_y = min(py, max_ny if max_ny % 2 == 1 else max_ny - 1)
    target_x = min(px, max_nx if max_nx % 2 == 1 else max_nx - 1)

    if (target_y, target_x) != (py, px):
        psf = center_crop(psf, (target_y, target_x))
        psf = crop_to_odd(psf)

    # Normalize PSF to sum to 1
    psf = psf / np.nansum(psf)
    return psf


def crop_to_odd(arr):
    """Center-crop a 2D array so both dimensions are odd."""
    ny, nx = arr.shape
    new_ny = ny if ny % 2 == 1 else ny - 1
    new_nx = nx if nx % 2 == 1 else nx - 1
    if (new_ny, new_nx) != (ny, nx):
        arr = center_crop(arr, (new_ny, new_nx))
    return arr


def center_crop(arr, target_shape):
    """Center-crop a 2D array to the given target shape."""
    ny, nx = arr.shape
    ty, tx = target_shape
    y0 = (ny - ty) // 2
    x0 = (nx - tx) // 2
    return arr[y0:y0 + ty, x0:x0 + tx]

In [85]:
def _first_existing_attr(obj, names):
    """Return (name, value) for the first attribute in `names` that exists
    and is not None on obj, else (None, None)."""
    for name in names:
        if hasattr(obj, name):
            val = getattr(obj, name)
            if val is not None:
                return name, val
    return None, None


def _resolve_position_guess(props, image_shape):
    """
    Get an (x, y) center guess from a SourceProperties instance, trying
    several attribute names since this isn't consistent across pysersic
    versions (e.g. some expose `position_guess`, others split it into
    separate x/y attributes, others only via the underlying `cat` catalog
    object from photutils' data_properties/SourceCatalog).
    """
    name, val = _first_existing_attr(
        props, ["position_guess", "pos_guess", "xy_guess"]
    )
    if val is not None:
        return float(val[0]), float(val[1])

    xname, xval = _first_existing_attr(props, ["x_guess", "xc_guess", "x0_guess"])
    yname, yval = _first_existing_attr(props, ["y_guess", "yc_guess", "y0_guess"])
    if xval is not None and yval is not None:
        return float(xval), float(yval)

    # Fall back to the underlying photutils catalog object, if present.
    cat = getattr(props, "cat", None)
    if cat is not None:
        for xattr, yattr in [("xcentroid", "ycentroid"), ("x_centroid", "y_centroid")]:
            if hasattr(cat, xattr) and hasattr(cat, yattr):
                xv, yv = getattr(cat, xattr), getattr(cat, yattr)
                try:
                    return float(np.asarray(xv).ravel()[0]), float(np.asarray(yv).ravel()[0])
                except Exception:
                    return float(xv), float(yv)

    # Last resort: image center.
    ny, nx = image_shape
    print("    [warning] Could not find a position guess on SourceProperties "
          "or its .cat catalog; falling back to the image center. Run "
          "inspect_source_properties(props) to see what's actually "
          "available on your installed version.")
    return float(nx // 2), float(ny // 2)


def inspect_source_properties(props):
    """
    Diagnostic: print every public, non-callable attribute on a
    SourceProperties instance and its value, so you can see exactly what
    your installed pysersic version calls things (position/flux/r_eff/sky
    guesses etc.) instead of guessing attribute names blind.

        from run_pysersic_fit import SourceProperties, inspect_source_properties
        props = SourceProperties(image, mask=mask)
        inspect_source_properties(props)
    """
    print("Public attributes on this SourceProperties instance:")
    for name in sorted(dir(props)):
        if name.startswith("_"):
            continue
        try:
            val = getattr(props, name)
        except Exception as e:
            print(f"  {name}: <error accessing: {e}>")
            continue
        if callable(val):
            continue
        print(f"  {name} = {val!r}")


def _get_prop_guesses(props, image_shape):
    flux_name, flux_guess = _first_existing_attr(props, ["flux_guess"])
    _, flux_guess_err = _first_existing_attr(props, ["flux_guess_err"])
    _, r_eff_guess = _first_existing_attr(props, ["r_eff_guess"])
    _, r_eff_guess_err = _first_existing_attr(props, ["r_eff_guess_err"])
    _, sky_guess = _first_existing_attr(props, ["sky_guess"])
    _, sky_guess_err = _first_existing_attr(props, ["sky_guess_err"])

    missing = [n for n, v in [
        ("flux_guess", flux_guess), ("flux_guess_err", flux_guess_err),
        ("r_eff_guess", r_eff_guess), ("r_eff_guess_err", r_eff_guess_err),
        ("sky_guess", sky_guess), ("sky_guess_err", sky_guess_err),
    ] if v is None]
    if missing:
        raise AttributeError(
            f"SourceProperties is missing expected attribute(s): {missing}. "
            f"Run inspect_source_properties(props) on your installed "
            f"pysersic version to find the correct attribute names, then "
            f"update _get_prop_guesses() accordingly."
        )

    xg, yg = _resolve_position_guess(props, image_shape)

    return dict(
        flux_guess=float(flux_guess),
        flux_guess_err=float(flux_guess_err),
        position_guess=(xg, yg),
        r_eff_guess=float(r_eff_guess),
        r_eff_guess_err=float(r_eff_guess_err),
        sky_guess=float(sky_guess),
        sky_guess_err=float(sky_guess_err),
    )


def _find_map_svi(model, model_kwargs, rkey, num_steps=6000, learning_rate=3e-2):
    """
    Quick MAP-like point estimate for the custom model, via SVI with an
    AutoDelta guide (equivalent to MAP under a flat reference measure).
    Mirrors the role of fitter.find_MAP() for the other, natively
    supported profiles.

    Uses init_to_median() rather than numpyro's default init_to_uniform():
    the default samples a random starting point in *unconstrained* space,
    which for tightly-constrained priors (e.g. flux, whose sigma is often
    only ~1-2% of its mean) can start optimization miles from anything
    sensible -- a classic cause of "converges to garbage" for a model like
    this one with a flux<->point-source-fraction degeneracy. init_to_median
    starts at each prior's median instead, which is centered on the
    SourceProperties-derived guesses -- a much saner starting point.
    """
    guide = AutoDelta(model, init_loc_fn=init_to_median())
    svi = SVI(model, guide, Adam(learning_rate), loss=Trace_ELBO())
    svi_state = svi.init(rkey, **model_kwargs)

    def body(state, _):
        state, loss = svi.update(state, **model_kwargs)
        return state, loss

    svi_state, losses = jax.lax.scan(body, svi_state, None, length=num_steps)
    params = svi.get_params(svi_state)
    map_params = {k.replace("_auto_loc", ""): float(v) for k, v in params.items()}
    losses = np.asarray(losses)
    print(f"    SVI loss: start={losses[0]:.4e}  end={losses[-1]:.4e}  "
          f"min={losses.min():.4e}")
    if not np.isfinite(losses[-1]):
        print("    [warning] Final SVI loss is not finite -- MAP estimate "
              "is unreliable. Check the printed guesses/priors below for "
              "anything degenerate (e.g. zero or negative error bars).")
    return map_params, float(losses[-1])

In [86]:
def statmorph_model(image, segmap, psf, rms):

    # ny, nx = image.shape
    # y, x = np.mgrid[0:ny, 0:nx]

    start = time.time()
    source_morphs = statmorph.source_morphology(
        image, segmap, psf=psf, weightmap=rms)
    print('Time: %g s.' % (time.time() - start))

    return source_morphs



In [87]:
def examining_output(id, filt, source_morphs, output_dir):

    os.makedirs(output_dir, exist_ok=True)
    
    morph = source_morphs[0]
    # source_morphs contains data on lost of objects in my image. Ext 0 ought to be my galaxy but maybe check.
    """
    print('BASIC MEASUREMENTS (NON-PARAMETRIC)')
    print('xc_centroid =', morph.xc_centroid)
    print('yc_centroid =', morph.yc_centroid)
    print('ellipticity_centroid =', morph.ellipticity_centroid)
    print('elongation_centroid =', morph.elongation_centroid)
    print('orientation_centroid =', morph.orientation_centroid)
    print('xc_asymmetry =', morph.xc_asymmetry)
    print('yc_asymmetry =', morph.yc_asymmetry)
    print('ellipticity_asymmetry =', morph.ellipticity_asymmetry)
    print('elongation_asymmetry =', morph.elongation_asymmetry)
    print('orientation_asymmetry =', morph.orientation_asymmetry)
    print('rpetro_circ =', morph.rpetro_circ)
    print('rpetro_ellip =', morph.rpetro_ellip)
    print('rhalf_circ =', morph.rhalf_circ)
    print('rhalf_ellip =', morph.rhalf_ellip)
    print('r20 =', morph.r20)
    print('r80 =', morph.r80)
    print('Gini =', morph.gini)
    print('M20 =', morph.m20)
    print('F(G, M20) =', morph.gini_m20_bulge)
    print('S(G, M20) =', morph.gini_m20_merger)
    print('sn_per_pixel =', morph.sn_per_pixel)
    print('C =', morph.concentration)
    print('A =', morph.asymmetry)
    print('S =', morph.smoothness)
    print()
    print('SERSIC MODEL')
    print('sersic_amplitude =', morph.sersic_amplitude)
    print('sersic_rhalf =', morph.sersic_rhalf)
    print('sersic_n =', morph.sersic_n)
    print('sersic_xc =', morph.sersic_xc)
    print('sersic_yc =', morph.sersic_yc)
    print('sersic_ellip =', morph.sersic_ellip)
    print('sersic_theta =', morph.sersic_theta)
    print('sersic_chi2_dof =', morph.sersic_chi2_dof)
    print()
    print('OTHER')
    print('sky_mean =', morph.sky_mean)
    print('sky_median =', morph.sky_median)
    print('sky_sigma =', morph.sky_sigma)
    print('flag =', morph.flag)
    print('flag_sersic =', morph.flag_sersic)
    """
    
    row = {
        'SURVEY_ID': id,
        f'{filt}_Gini': morph.gini,
        f'{filt}_M20 ': morph.m20,
        f'{filt}_F(G, M20)': morph.gini_m20_bulge,
        f'{filt}_S(G, M20)': morph.gini_m20_merger,
        f'{filt}_C': morph.concentration,
        f'{filt}_A': morph.asymmetry,
        f'{filt}_S': morph.smoothness,
        f'{filt}_flag': morph.flag,
        f'{filt}_sersic_n': morph.sersic_n,
        f'{filt}_flag_sersic': morph.flag_sersic
    }

    fig = make_figure(morph)
    fig_path = os.path.join(output_dir, f"{filt}_statmorph.png")
    fig.savefig(fig_path, dpi=150)
    plt.close(fig)

    return row

In [88]:
def run_everything():

    rows = []

    for i in range(len(GALAXY_ID)):
        for filt in FILTERS:

            try:
                
                image, mask, rms, segmap = load_science_data(GALAXY_ID[i], filt)

                # Path to the PSF FITS file, psf will be cropped to match/fit the science image.
                psf_fits_path = f"/nvme/scratch/work/alberttg/Summer_project/Cutouts_3.0as/{GALAXY_ID[i]}/cutouts/{GALAXY_ID[i]}_psf_{filt}.fits"
                # psf = load_and_crop_psf(psf_fits_path, image.shape, psf_ext=0)
                psf = read_fits_path(psf_fits_path, ext=0)

                # Output directory for plots/results
                output_dir = f"/nvme/scratch/work/alberttg/Summer_project/Statmorph_fits/{GALAXY_ID[i]}"

                # sky_pixels = rms[segmap == 0]
                # sky_sigma = np.median(sky_pixels)

                source_morphs = statmorph_model(image, segmap, psf, rms)
                row = examining_output(GALAXY_ID[i], filt, source_morphs, output_dir)
                rows.append(row)

            except Exception as e:
                print(e)
                continue
    
    statmorph_table = Table(rows=rows)
    statmorph_table.write("Statmorph_table.fits", format="fits", overwrite=True)

In [89]:
if __name__ == "__main__":
    run_everything()

/nvme/scratch/software/anaconda3/envs/statmorph_env/lib/python3.10/site-packages/statmorph/utils/image_diagnostics.py:293: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc=4, fontsize=12, facecolor='w', framealpha=1.0, edgecolor='k')


Time: 3.92754 s.


/nvme/scratch/software/anaconda3/envs/statmorph_env/lib/python3.10/site-packages/statmorph/utils/image_diagnostics.py:293: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc=4, fontsize=12, facecolor='w', framealpha=1.0, edgecolor='k')


Time: 4.24838 s.


Time: 5.01116 s.


...

/nvme/scratch/software/anaconda3/envs/statmorph_env/lib/python3.10/site-packages/statmorph/utils/image_diagnostics.py:293: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc=4, fontsize=12, facecolor='w', framealpha=1.0, edgecolor='k')


Time: 1.88264 s.


/nvme/scratch/software/anaconda3/envs/statmorph_env/lib/python3.10/site-packages/statmorph/utils/image_diagnostics.py:293: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc=4, fontsize=12, facecolor='w', framealpha=1.0, edgecolor='k')


Time: 0.550577 s.
Time: 0.631409 s.
Time: 0.521237 s.


KeyboardInterrupt: 